In [9]:
!pip install pandas numpy scikit-learn matplotlib jupyter

Defaulting to user installation because normal site-packages is not writeable
  Using cached ipykernel-7.4.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached comm-0.2.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached debugpy-1.8.22-cp314-cp314-win_amd64.whl.metadata (1.4 kB)
  Using cached ipython-9.18.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached jupyter_client-8.10.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyter_core-5.9.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached matplotlib_inline-0.2.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached nest_asyncio2-1.7.4-py3-none-any.whl.metadata (7.0 kB)
  Using cached pyzmq-27.3.0-cp312-abi3-win_amd64.whl.metadata (3.8 kB)
  Using cached tornado-6.6-cp311-abi3-win_amd64.whl.metadata (2.7 kB)
  Using cached traitlets-5.16.1-py3-none-any.whl.metadata (10 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached ipython_pygments_lexers-1.1.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached j

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.6 MB 375.7 kB/s eta 0:00:14
   --- ------------------------------------ 0.5/5.6 MB 375.7 kB/s eta 0:00:14
   --- ------------------------------------ 0.5/5.6 MB 375.7 kB/s eta 0:00:14
   ----- ---------------------------------- 0.8/5.6 MB 398.9 kB/s eta 0:00:13
   ----- ---------------------------------- 0.8/5.6 MB 398.9 kB/s eta 0:00:13
   ------- -------------------------------- 1.0/5.6 MB 454.6 kB/s eta 0:00:11
   ------- -------------------------------- 1.0/5

In [2]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

DATA_RAW_DIR = "../mimic-iv-clinical-database-demo-2.2/hosp" # Hoặc đường dẫn thư mục demo của em
DATA_PROCESSED_DIR = "../data/processed"

os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
print("Thư viện và thư mục đã sẵn sàng!")

ModuleNotFoundError: No module named 'pandas'

In [3]:
df_adm = pd.read_csv(
    os.path.join(DATA_RAW_DIR, "admissions.csv"),
    usecols=["subject_id", "hadm_id", "admittime"]
)

df_diag = pd.read_csv(
    os.path.join(DATA_RAW_DIR, "diagnoses_icd.csv"),
    usecols=["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"]
)

df_adm["admittime"] = pd.to_datetime(df_adm["admittime"])

print(f"Tổng số lượt nhập viện: {len(df_adm)}")
print(f"Tổng số hồ sơ chẩn đoán: {len(df_diag)}")

NameError: name 'pd' is not defined

In [4]:
df_diag_icd9 = df_diag[df_diag["icd_version"] == 9].copy()

df_diag_icd9.dropna(subset=["icd_code"], inplace=True)
df_diag_icd9["icd_code"] = df_diag_icd9["icd_code"].astype(str).str.strip()

visit_counts = df_diag_icd9.groupby("subject_id")["hadm_id"].nunique()

valid_patients = visit_counts[visit_counts >= 2].index.tolist()

print(f"Số lượng bệnh nhân có >= 2 lượt khám (ICD-9): {len(valid_patients)}")

df_filtered_diag = df_diag_icd9[df_diag_icd9["subject_id"].isin(valid_patients)]

NameError: name 'df_diag' is not defined

In [5]:

df_merged = pd.merge(
    df_filtered_diag, 
    df_adm[["hadm_id", "admittime"]], 
    on="hadm_id", 
    how="inner"
)

df_merged.sort_values(
    by=["subject_id", "admittime", "seq_num"], 
    ascending=[True, True, True], 
    inplace=True
)

print("Ví dụ 5 dòng dữ liệu sau khi sắp xếp:")
display(df_merged.head())

NameError: name 'pd' is not defined

In [6]:
def icd9_to_ccs_category(icd_code):
    
    clean_code = icd_code.replace(".", "").upper()
    
    prefix = clean_code[:3]
    
    if prefix.isdigit():
        num = int(prefix)
        if 390 <= num <= 459:
            return "CCS_CIRCULATORY"  # Bệnh hệ tuần hoàn (Tim mạch, Huyết áp...)
        elif 460 <= num <= 519:
            return "CCS_RESPIRATORY"  # Bệnh hệ hô hấp
        elif 520 <= num <= 579:
            return "CCS_DIGESTIVE"    # Bệnh hệ tiêu hóa
        elif 240 <= num <= 279:
            return "CCS_ENDOCRINE"    # Bệnh nội tiết, chuyển hóa [8]
        elif 580 <= num <= 629:
            return "CCS_GENITOURINARY" # Bệnh tiết niệu/sinh dục
    
    return "CCS_OTHER"

df_merged["ccs_code"] = df_merged["icd_code"].apply(icd9_to_ccs_category)

NameError: name 'df_merged' is not defined

In [7]:
patient_sequences = {}

for subject_id, group_patient in df_merged.groupby("subject_id", sort=False):
    patient_visits = []
    
    for hadm_id, group_visit in group_patient.groupby("hadm_id", sort=False):
        icd_list = group_visit["icd_code"].unique().tolist()
        patient_visits.append(icd_list)
    
    patient_sequences[str(subject_id)] = patient_visits

sample_patient_id = list(patient_sequences.keys())
print(f"Cấu trúc chuỗi khám của Bệnh nhân ID {sample_patient_id}:")
print(json.dumps(patient_sequences[sample_patient_id], indent=2))

NameError: name 'df_merged' is not defined

In [8]:
patient_ids = list(patient_sequences.keys())

train_ids, test_ids = train_test_split(patient_ids, test_size=0.20, random_state=42)

train_ids, val_ids = train_test_split(train_ids, test_size=0.15, random_state=42)

print(f"Số bệnh nhân tập Train: {len(train_ids)}")
print(f"Số bệnh nhân tập Validation: {len(val_ids)}")
print(f"Số bệnh nhân tập Test: {len(test_ids)}")

processed_data = {
    "sequences": patient_sequences,
    "splits": {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids
    }
}

output_filepath = os.path.join(DATA_PROCESSED_DIR, "patient_sequences.json")
with open(output_filepath, "w") as f:
    json.dump(processed_data, f, indent=2)

print(f"\n✅ Đã lưu thành công dữ liệu tiền xử lý vào: {output_filepath}")

NameError: name 'train_test_split' is not defined